# Olist E-Commerce — ML Model
### Predicting Delivery Delays

ZAKA Business Analytics Capstone  

Ayah Miqdady
Fadwa
Basma Badidi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Capstone project/data/'


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              classification_report, confusion_matrix, roc_auc_score, roc_curve)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)



## 1. Load the data

Olist gives us 9 separate CSV files. We only need a few of them for the ML model.  
The key ones are orders (timestamps), order items (price, freight), products (category), customers (location), and reviews (score).

In [ ]:
orders    = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
items     = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
products  = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
customers = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
reviews   = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
payments  = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
category_translation = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

print('orders:   ', orders.shape)
print('items:    ', items.shape)
print('products: ', products.shape)
print('customers:', customers.shape)
print('reviews:  ', reviews.shape)
print('payments: ', payments.shape)

In [ ]:
# quick look at what the orders table gives us — the timestamps are what we care about most
orders.head(3)

## 2. Build the target variable — delivery delay

Delivery delay = actual delivery date minus estimated delivery date.  
Positive means it arrived late, negative means it arrived early.  
We'll use this two ways: as a regression target (how many days late?) and as a classification target (late or not late?).

In [ ]:
# convert all date columns from string to actual datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

# only keep orders that were actually delivered — can't measure delay otherwise
delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f'delivered orders: {len(delivered)} out of {len(orders)} total')

In [ ]:
# the main target — delay in days
delivered['delivery_delay_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_estimated_delivery_date']
).dt.days

# binary target: 1 = late, 0 = on time or early
delivered['is_late'] = (delivered['delivery_delay_days'] > 0).astype(int)

# also useful: how long did approval take after purchase?
delivered['approval_time_hours'] = (
    delivered['order_approved_at'] - delivered['order_purchase_timestamp']
).dt.total_seconds() / 3600

# how long was the estimated delivery window?
delivered['estimated_window_days'] = (
    delivered['order_estimated_delivery_date'] - delivered['order_purchase_timestamp']
).dt.days

# day of week and month of purchase — useful time features
delivered['purchase_dayofweek'] = delivered['order_purchase_timestamp'].dt.dayofweek
delivered['purchase_month']     = delivered['order_purchase_timestamp'].dt.month

print('late orders:', delivered['is_late'].sum())
print('on-time orders:', (delivered['is_late'] == 0).sum())
print(f'late rate: {delivered["is_late"].mean():.2%}')

In [ ]:
# distribution of delay days — good to see before modeling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(delivered['delivery_delay_days'].clip(-30, 60), bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', label='on time')
axes[0].set_title('Distribution of Delivery Delay (days)')
axes[0].set_xlabel('Days (positive = late)')
axes[0].legend()

delivered['is_late'].value_counts().plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'])
axes[1].set_title('Late vs On-Time Orders')
axes[1].set_xticklabels(['On Time (0)', 'Late (1)'], rotation=0)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Merge in the other tables and build features

We need price, freight, product category, customer state, payment value, and review score.  
Review score is technically a target we could predict separately, but here we'll use it as context.

In [ ]:
# aggregate items per order — some orders have multiple items
items_agg = items.groupby('order_id').agg(
    num_items       = ('order_item_id', 'count'),
    total_price     = ('price', 'sum'),
    total_freight   = ('freight_value', 'sum'),
    avg_price       = ('price', 'mean')
).reset_index()

# aggregate payments per order
payments_agg = payments.groupby('order_id').agg(
    payment_value        = ('payment_value', 'sum'),
    payment_installments = ('payment_installments', 'max'),
    payment_type         = ('payment_type', 'first')
).reset_index()

# most common product category per order
items_with_category = items.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
items_with_category = items_with_category.merge(category_translation, on='product_category_name', how='left')
top_category = items_with_category.groupby('order_id')['product_category_name_english'].first().reset_index()
top_category.columns = ['order_id', 'top_category']

# average review score per order
reviews_agg = reviews.groupby('order_id')['review_score'].mean().reset_index()

print('aggregations done')

In [ ]:
# now merge everything into one dataframe
df = delivered[[
    'order_id', 'customer_id',
    'delivery_delay_days', 'is_late',
    'approval_time_hours', 'estimated_window_days',
    'purchase_dayofweek', 'purchase_month'
]].copy()

df = df.merge(items_agg,      on='order_id', how='left')
df = df.merge(payments_agg,   on='order_id', how='left')
df = df.merge(top_category,   on='order_id', how='left')
df = df.merge(reviews_agg,    on='order_id', how='left')
df = df.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')

print('merged shape:', df.shape)
df.head(3)

In [ ]:
# check nulls before doing anything else
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0])

In [ ]:
# fill nulls — most are categorical, so fill with 'unknown' or median for numerics

df['top_category']   = df['top_category'].fillna('unknown')
df['payment_type']   = df['payment_type'].fillna('unknown')
df['customer_state'] = df['customer_state'].fillna('unknown')

# for numeric columns, fill with median — safer than mean when there are outliers
numeric_cols = ['delivery_delay_days', 'approval_time_hours', 'estimated_window_days', 'total_price',
                'total_freight', 'avg_price', 'payment_value', 'payment_installments', 'review_score']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

print('nulls remaining:', df.isnull().sum().sum())


Before touching any model, we need to actually understand the data.  

### 4a. Correlation Analysis

We want to know which numeric variables are actually related to delivery delay.  
High correlation with the target = useful feature. High correlation between two features = possible redundancy.

In [ ]:
# build the working dataframe before encoding so we can correlate on raw numbers
# we'll do encoding in the next section — this is just for analysis

numeric_for_corr = [
    'delivery_delay_days', 'is_late',
    'approval_time_hours', 'estimated_window_days',
    'purchase_dayofweek', 'purchase_month',
    'num_items', 'total_price', 'total_freight',
    'avg_price', 'payment_value', 'payment_installments',
    'review_score'
]

corr_df = df[numeric_for_corr].dropna()
corr_matrix = corr_df.corr()

plt.figure(figsize=(13, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # only show lower triangle
sns.heatmap(
    corr_matrix, mask=mask,
    annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Correlation Matrix — All Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# pull out just the correlations with the target variable, sorted
target_corr = corr_matrix['delivery_delay_days'].drop('delivery_delay_days').sort_values(key=abs, ascending=False)

print('Correlation with delivery_delay_days (absolute value, descending):')
print(target_corr.round(4).to_string())
print()
print('Positive = tends to increase delay')
print('Negative = tends to reduce delay')

In [ ]:
# bar chart version — easier to read than the number table above
plt.figure(figsize=(10, 6))
colors = ['tomato' if v > 0 else 'steelblue' for v in target_corr.values]
target_corr.plot(kind='barh', color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Delivery Delay')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.show()

In [ ]:
# scatter plots for the top 4 most correlated features vs delay
top_features = target_corr.abs().nlargest(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].scatter(
        corr_df[feat],
        corr_df['delivery_delay_days'],
        alpha=0.2, s=5, color='steelblue'
    )
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('delivery_delay_days')
    axes[i].set_title(f'{feat} vs delay (r={corr_matrix.loc[feat, "delivery_delay_days"]:.2f})')
    axes[i].axhline(0, color='red', linestyle='--', linewidth=0.8)

plt.suptitle('Top Correlated Features vs Delivery Delay', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 4b. Outlier Detection

Outliers can mess up both the analysis and the model if we don't know they're there.  
We'll use boxplots to spot them visually and IQR to quantify them.

In [ ]:
# boxplots for all key numeric features — outliers show up as dots beyond the whiskers
outlier_cols = ['delivery_delay_days', 'total_price', 'total_freight',
                'approval_time_hours', 'estimated_window_days', 'payment_value']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(outlier_cols):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(col)
    axes[i].set_xticks([])

plt.suptitle('Boxplots — Outlier Detection', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# IQR method — count how many outliers each column has and what percentage of the data they are
print('Outlier counts using IQR method (values beyond 1.5 * IQR):')
print(f'{"Column":<30} {"Outliers":>10} {"% of data":>12}')
print('-' * 55)

for col in outlier_cols:
    series = df[col].dropna()
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((series < lower) | (series > upper)).sum()
    pct = n_outliers / len(series) * 100
    print(f'{col:<30} {n_outliers:>10,} {pct:>11.2f}%')

In [ ]:
# look at the extreme delivery delay cases specifically
# these could be genuine very late orders or data errors
extreme_late = df[df['delivery_delay_days'] > 30]
extreme_early = df[df['delivery_delay_days'] < -20]

print(f'Orders more than 30 days late:  {len(extreme_late):,} ({len(extreme_late)/len(df)*100:.2f}%)')
print(f'Orders more than 20 days early: {len(extreme_early):,} ({len(extreme_early)/len(df)*100:.2f}%)')
print()
print('Sample of extreme late orders:')
print(extreme_late[['delivery_delay_days', 'total_price', 'total_freight', 'customer_state']].head(5))

### 4c. Key Feature Visualizations


In [ ]:
# average delay by month — is there a seasonal pattern?
monthly_delay = df.groupby('purchase_month')['delivery_delay_days'].mean()

plt.figure(figsize=(10, 5))
monthly_delay.plot(kind='bar', color='steelblue', edgecolor='white')
plt.axhline(0, color='red', linestyle='--', linewidth=0.8, label='on time')
plt.title('Average Delivery Delay by Purchase Month')
plt.xlabel('Month')
plt.ylabel('Avg Delay (days)')
plt.xticks(ticks=range(12),
           labels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
           rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# late rate by customer state — some regions are clearly worse than others
state_late = df.groupby('customer_state')['is_late'].agg(['mean', 'count']).reset_index()
state_late.columns = ['state', 'late_rate', 'order_count']
state_late = state_late[state_late['order_count'] > 100]  # ignore states with very few orders
state_late = state_late.sort_values('late_rate', ascending=False)

plt.figure(figsize=(14, 5))
bars = plt.bar(state_late['state'], state_late['late_rate'], color='steelblue', edgecolor='white')
plt.axhline(df['is_late'].mean(), color='red', linestyle='--', label=f'overall avg ({df["is_late"].mean():.2%})')
plt.title('Late Delivery Rate by Customer State')
plt.xlabel('State')
plt.ylabel('Late Rate')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# top 15 product categories by late rate — which categories are the worst offenders?
cat_late = df.groupby('top_category')['is_late'].agg(['mean', 'count']).reset_index()
cat_late.columns = ['category', 'late_rate', 'order_count']
cat_late = cat_late[cat_late['order_count'] > 200]  # need enough orders to be meaningful
cat_late = cat_late.sort_values('late_rate', ascending=False).head(15)

plt.figure(figsize=(12, 6))
sns.barplot(data=cat_late, x='late_rate', y='category', palette='Reds_r')
plt.axvline(df['is_late'].mean(), color='navy', linestyle='--', label='overall avg')
plt.title('Top 15 Product Categories by Late Delivery Rate')
plt.xlabel('Late Rate')
plt.ylabel('')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# review score vs late — do late deliveries actually get worse reviews?
review_by_late = df.groupby('is_late')['review_score'].value_counts(normalize=True).unstack()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# average score comparison
df.groupby('is_late')['review_score'].mean().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white'
)
axes[0].set_title('Average Review Score: On Time vs Late')
axes[0].set_xticklabels(['On Time', 'Late'], rotation=0)
axes[0].set_ylabel('Avg Review Score')
axes[0].set_ylim(0, 5)

# distribution of review scores for each group
df[df['is_late'] == 0]['review_score'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='steelblue', alpha=0.6, label='On Time', edgecolor='white'
)
df[df['is_late'] == 1]['review_score'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='tomato', alpha=0.6, label='Late', edgecolor='white'
)
axes[1].set_title('Review Score Distribution: On Time vs Late')
axes[1].set_xlabel('Review Score')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# freight cost vs delay — heavier/more expensive shipments take longer?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# freight vs delay days (capped to remove extreme outliers visually)
axes[0].scatter(
    df['total_freight'].clip(0, 100),
    df['delivery_delay_days'].clip(-20, 40),
    alpha=0.15, s=5, color='steelblue'
)
axes[0].axhline(0, color='red', linestyle='--', linewidth=0.8)
axes[0].set_xlabel('Total Freight Value (capped at 100)')
axes[0].set_ylabel('Delivery Delay (days)')
axes[0].set_title('Freight Cost vs Delivery Delay')

# estimated delivery window vs actual delay
axes[1].scatter(
    df['estimated_window_days'].clip(0, 60),
    df['delivery_delay_days'].clip(-20, 40),
    alpha=0.15, s=5, color='darkorange'
)
axes[1].axhline(0, color='red', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('Estimated Delivery Window (days)')
axes[1].set_ylabel('Delivery Delay (days)')
axes[1].set_title('Estimated Window vs Actual Delay')

plt.tight_layout()
plt.show()

### 4d. Key Findings from Exploration

Before moving to modeling, write down what the data actually told us.  
These findings should go into the report and the presentation.

In [ ]:
# summary stats — the numbers behind the findings
print('=== KEY FINDINGS SUMMARY ===')
print()

print(f'Overall late rate: {df["is_late"].mean():.2%}')
print(f'Avg delay when late: {df[df["is_late"]==1]["delivery_delay_days"].mean():.1f} days')
print(f'Avg review score on-time: {df[df["is_late"]==0]["review_score"].mean():.2f}')
print(f'Avg review score late:    {df[df["is_late"]==1]["review_score"].mean():.2f}')
print()

print('Top 3 strongest correlates with delivery delay:')
top3 = target_corr.abs().nlargest(3)
for feat, val in top3.items():
    direction = 'positive' if target_corr[feat] > 0 else 'negative'
    print(f'  {feat}: r={target_corr[feat]:.3f} ({direction})')
print()

print('Worst state for late deliveries:')
worst_state = state_late.iloc[0]
print(f'  {worst_state["state"]} — {worst_state["late_rate"]:.2%} late rate ({worst_state["order_count"]:,} orders)')
print()

print('Outlier summary:')
print(f'  Orders > 30 days late: {len(extreme_late):,}')
print(f'  Orders > 20 days early: {len(extreme_early):,}')
print()
print('These findings guide our feature selection and model design in the next section.')

---
**Findings to carry into the model:**
- estimated_window_days and approval_time_hours are likely the strongest predictors — orders with longer windows and slower approvals tend to be later
- Certain states and product categories have significantly higher late rates — these encoded features should matter to the model
- Late orders receive noticeably lower review scores, confirming delay has real business impact
- Extreme outliers (>30 days late) exist but are a small fraction — we'll cap the regression target when training
- No single feature has a very high correlation, which means the problem benefits from a non-linear model (XGBoost) over linear regression

## 4. Encode categorical features

The model can't read strings, so we need to convert customer_state, top_category, and payment_type into numbers.  
We'll use label encoding here — simple and works fine for tree-based models.

In [ ]:
le = LabelEncoder()
cat_cols = ['top_category', 'payment_type', 'customer_state']

for col in cat_cols:
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))

# final feature list for the model
feature_cols = [
    'num_items', 'total_price', 'total_freight', 'avg_price',
    'payment_value', 'payment_installments',
    'approval_time_hours', 'estimated_window_days',
    'purchase_dayofweek', 'purchase_month',
    'top_category_encoded', 'payment_type_encoded', 'customer_state_encoded'
]

# Ensure no NaNs remain in relevant columns before creating X and y
cols_to_check = feature_cols + ['delivery_delay_days', 'is_late'] # Include 'is_late' for classification target consistency
initial_rows = len(df)
df = df.dropna(subset=cols_to_check).copy()
print(f'Dropped {initial_rows - len(df)} rows with NaNs in feature or target columns.')

X = df[feature_cols]
y_class = df['is_late']              # for classification
y_reg   = df['delivery_delay_days']  # for regression

print('features:', X.shape)
print('class balance:\n', y_class.value_counts())

## 5. Train/test split

80/20 split. Stratified on the class label so both splits have the same late/on-time ratio.

In [ ]:
X_train, X_test, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

_, _, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print('train size:', X_train.shape[0])
print('test size: ', X_test.shape[0])

## 6. Baseline model — Logistic Regression

In [ ]:
scaler   = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train_c)

lr_preds = lr.predict(X_test_scaled)
print('Baseline — Logistic Regression')
print(classification_report(y_test_c, lr_preds, target_names=['On Time', 'Late']))
print('ROC-AUC:', roc_auc_score(y_test_c, lr.predict_proba(X_test_scaled)[:, 1]).round(4))

## 7. Random Forest — Classification

Tree-based models usually handle this kind of tabular data better than linear models.  
Also gives us feature importance for free.

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train_c)

rf_preds = rf_clf.predict(X_test)
rf_proba = rf_clf.predict_proba(X_test)[:, 1]

print('Random Forest — Classification')
print(classification_report(y_test_c, rf_preds, target_names=['On Time', 'Late']))
print('ROC-AUC:', roc_auc_score(y_test_c, rf_proba).round(4))

In [ ]:
# confusion matrix — visual is clearer than numbers alone
cm = confusion_matrix(y_test_c, rf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['On Time', 'Late'],
            yticklabels=['On Time', 'Late'])
plt.title('Random Forest — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test_c, rf_proba)
auc_score   = roc_auc_score(y_test_c, rf_proba)

plt.plot(fpr, tpr, label=f'Random Forest (AUC = {auc_score:.3f})', color='steelblue')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

## 8. XGBoost — Classification

XGBoost usually beats Random Forest on structured data.  
We'll compare the two and keep whichever performs better.

In [ ]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_clf.fit(X_train, y_train_c)

xgb_preds = xgb_clf.predict(X_test)
xgb_proba = xgb_clf.predict_proba(X_test)[:, 1]

print('XGBoost — Classification')
print(classification_report(y_test_c, xgb_preds, target_names=['On Time', 'Late']))
print('ROC-AUC:', roc_auc_score(y_test_c, xgb_proba).round(4))

In [ ]:
# side by side ROC comparison
fpr_rf,  tpr_rf,  _ = roc_curve(y_test_c, rf_proba)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test_c, xgb_proba)

plt.plot(fpr_rf,  tpr_rf,  label=f'Random Forest (AUC = {roc_auc_score(y_test_c, rf_proba):.3f})')
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost       (AUC = {roc_auc_score(y_test_c, xgb_proba):.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

## 9. Feature Importance

This tells us which inputs actually mattered to the model.  
Important for the presentation — stakeholders always want to know what's driving the predictions.

In [ ]:
importance_df = pd.DataFrame({
    'feature':   feature_cols,
    'importance': xgb_clf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='importance', y='feature', palette='Blues_r')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## 10. SHAP Values — explaining what the model actually learned

Feature importance tells you which features mattered overall.  
SHAP goes deeper — it shows how each feature pushed individual predictions up or down.  
This is the interpretability piece we promised in the proposal.

In [ ]:
try:
    import shap
    shap.initjs()

    explainer   = shap.TreeExplainer(xgb_clf)
    shap_values = explainer.shap_values(X_test)

    # summary plot — shows direction and magnitude of each feature
    plt.figure()
    shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=True)

except ImportError:
    print('shap not installed — run: pip install shap')
    print('skipping SHAP plots for now')

## 11. Regression model — predicting delay in days (not just late/on time)

Classification tells us *if* an order will be late.  
Regression tells us *how* late — which is more useful operationally.

In [ ]:
# align regression target with the same X_train / X_test split
y_train_r_aligned = y_reg.loc[X_train.index]
y_test_r_aligned  = y_reg.loc[X_test.index]

# clean training target
train_mask = y_train_r_aligned.notna() & y_train_r_aligned.between(-30, 60)
X_train_r = X_train.loc[train_mask]
y_train_r_clean = y_train_r_aligned.loc[train_mask]

xgb_reg.fit(X_train_r, y_train_r_clean)

y_pred_reg = xgb_reg.predict(X_test)

# clean evaluation target
test_mask = y_test_r_aligned.notna()

mae  = mean_absolute_error(y_test_r_aligned[test_mask], y_pred_reg[test_mask])
rmse = np.sqrt(mean_squared_error(y_test_r_aligned[test_mask], y_pred_reg[test_mask]))
r2   = r2_score(y_test_r_aligned[test_mask], y_pred_reg[test_mask])

print(f"MAE:  {mae:.2f} days")
print(f"RMSE: {rmse:.2f} days")
print(f"R²:   {r2:.4f}")

In [ ]:
# actual vs predicted — a scatter plot is a clean way to show how well the regression did
plt.figure(figsize=(8, 6))
plt.scatter(y_test_r, y_pred_reg, alpha=0.3, color='steelblue', s=10)
plt.plot([y_test_r.min(), y_test_r.max()],
         [y_test_r.min(), y_test_r.max()],
         'r--', label='perfect prediction')
plt.xlabel('Actual Delay (days)')
plt.ylabel('Predicted Delay (days)')
plt.title('Regression: Actual vs Predicted Delivery Delay')
plt.legend()
plt.tight_layout()
plt.show()

## 12. Cross-validation

A single train/test split can be lucky or unlucky depending on which rows end up where.  
Cross-validation runs the model 5 times on different splits and gives us a more honest accuracy estimate.

In [ ]:
cv_scores = cross_val_score(
    xgb_clf, X, y_class,
    cv=5, scoring='roc_auc', n_jobs=-1
)

print('5-fold Cross-Validation ROC-AUC scores:')
for i, score in enumerate(cv_scores):
    print(f'  Fold {i+1}: {score:.4f}')
print(f'  Mean:   {cv_scores.mean():.4f}')
print(f'  Std:    {cv_scores.std():.4f}')
print()
print('Low std means the model is consistent, not just getting lucky on one split.')

## 13. Model summary

Pulling everything together before we move to the dashboard and roadmap.

In [ ]:
results = {
    'Model': ['Logistic Regression (baseline)', 'Random Forest', 'XGBoost (final)'],
    'ROC-AUC': [
        round(roc_auc_score(y_test_c, lr.predict_proba(X_test_scaled)[:, 1]), 4),
        round(roc_auc_score(y_test_c, rf_proba), 4),
        round(roc_auc_score(y_test_c, xgb_proba), 4)
    ]
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print()
print('Final model: XGBoost classifier')
print(f'Regression MAE: {mae:.2f} days  |  RMSE: {rmse:.2f} days  |  R²: {r2:.4f}')
print(f'Cross-val AUC:  {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

## 14. Save outputs for the dashboard


In [ ]:
# build an export dataframe — test set + predictions
export_df = X_test.copy()
export_df['actual_is_late']       = y_test_c.values
export_df['predicted_is_late']    = xgb_preds
export_df['late_probability']     = xgb_proba.round(4)
export_df['actual_delay_days'] = y_test_r_aligned.values
export_df['predicted_delay_days'] = y_pred_reg.round(1)

export_df.to_csv('model_predictions.csv', index=False)
print('saved model_predictions.csv')
print(f'{len(export_df)} rows')
export_df.head()